In [1]:
# Calculate PM25 population weighted exposure

In [1]:
import os
import xarray as xr
import numpy as np

In [2]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"
PM_DIR = "/glade/work/awells/air_quality/CESM/pm25/pm25_bc/"

In [3]:
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop_ssp2 = xr.open_dataarray(pop_path)

mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASK_DIR, mask_file)
country_mask = xr.open_dataarray(mask_path)

In [4]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/pm25/exposure/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        # Load data array
        if scenario == "ARISE":
            dates = "2035-2069"
        elif scenario == "SSP245":
            dates = "2020-2069"

        pm25_file = f"PM25_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        pm25_path = os.path.join(PM_DIR, pm25_file)
        pm25 = xr.open_dataarray(pm25_path)
        # Select same years as pm25 data for population
        population = pop_ssp2.sel(year=pm25.year)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        pm25 = pm25.reindex_like(country_mask, method="nearest",
                                 tolerance=1e-9, fill_value=0)
        population = population.reindex_like(country_mask,
                                             method="nearest", tolerance=1e-9)

        weighted_value = population * pm25

        country_list = []

        for i in range(204):
            print(f"Processing country number {i}")
            mask = country_mask.isel(country=i)
            country_weight = xr.where(
                mask == 1,
                weighted_value,
                np.nan).sum(dim=("lat", "lon"))
            pop_country = xr.where(
                mask == 1,
                population,
                np.nan).sum(dim=("lat", "lon"))
            country_pop_weighted = country_weight / pop_country
            country_list.append(country_pop_weighted)

        pop_weighted_exposure = xr.concat(country_list, "country")

        out_file = f"PM25_country_population_weighted_exposure_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        pop_weighted_exposure.to_netcdf(out_path)

Processing SSP245, Ensemble 08
Processing country number 0
Processing country number 1
Processing country number 2
Processing country number 3
Processing country number 4
Processing country number 5
Processing country number 6
Processing country number 7
Processing country number 8
Processing country number 9
Processing country number 10
Processing country number 11
Processing country number 12
Processing country number 13
Processing country number 14
Processing country number 15
Processing country number 16
Processing country number 17
Processing country number 18
Processing country number 19
Processing country number 20
Processing country number 21
Processing country number 22
Processing country number 23
Processing country number 24
Processing country number 25
Processing country number 26
Processing country number 27
Processing country number 28
Processing country number 29
Processing country number 30
Processing country number 31
Processing country number 32
Processing country num